# 07. Graph Spectral Null vs Torus Null

This notebook is standalone and brings in the required inputs from earlier notebooks.

It computes **spectral-null p-values** for edge-gradient couplings and compares them to
**torus-null p-values** exported by notebook 03.


In [ ]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "GSE238264_RAW").exists() or (_root / "st_adata_scored.h5ad").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# Run-mode toggle: set HCC_RUN_MODE=smoke for quick checks.
RUN_MODE = os.environ.get("HCC_RUN_MODE", "full").strip().lower()
N_PERM = 500 if RUN_MODE == "full" else 100
N_EIGS_MAX = 100
SEED = 7

# Load st_adata from memory or disk.
if "st_adata" not in globals():
    h5_path = Path("st_adata_scored.h5ad")
    if not h5_path.exists():
        raise FileNotFoundError(
            "st_adata_scored.h5ad not found. Run 02_visium_spot_scoring.ipynb first."
        )
    st_adata = sc.read_h5ad(h5_path.as_posix())

required_obs_cols = ["sample", "group"] if "group" in st_adata.obs.columns else ["sample"]
for c in required_obs_cols:
    if c not in st_adata.obs.columns:
        raise KeyError(f"st_adata.obs is missing required column: {c}")

needed_features = ["spot_exhaustion", "spot_cytotoxic", "spot_tam"]
missing = [c for c in needed_features if c not in st_adata.obs.columns]
if missing:
    raise KeyError(f"st_adata.obs missing required features: {missing}")

st_adata.obs["sample"] = st_adata.obs["sample"].astype(str)

# Build per-sample object dictionary.
st_by_sample = {
    s: st_adata[st_adata.obs["sample"] == s].copy()
    for s in sorted(st_adata.obs["sample"].unique())
}

# Build sample_groups robustly.
sample_groups = {}
if "response" in st_adata.obs.columns:
    tmp = st_adata.obs[["sample", "response"]].dropna().copy()
    tmp["response"] = tmp["response"].astype(str).str.strip().str.upper()
    m = {
        "R": "R", "RESPONDER": "R",
        "NR": "NR", "NONRESPONDER": "NR", "NON-RESPONDER": "NR", "NON_RESPONDER": "NR",
    }
    first_resp = tmp.groupby("sample", as_index=True)["response"].first()
    for s, r in first_resp.items():
        sample_groups[s] = m.get(r, r)

for s in st_by_sample:
    if s not in sample_groups:
        up = s.upper()
        sample_groups[s] = "NR" if up.endswith("NR") else ("R" if up.endswith("R") else None)

print("Samples:", sorted(st_by_sample.keys()))
print("Using N_PERM:", N_PERM)


In [ ]:
def graph_laplacian(W):
    """Return unnormalized graph Laplacian L = D - W."""
    if not sp.issparse(W):
        W = sp.csr_matrix(W)
    degrees = np.asarray(W.sum(axis=1)).ravel()
    D = sp.diags(degrees)
    return D - W


def edge_index_from_W(W):
    """Return unique undirected edges (i<j) from sparse adjacency."""
    coo = W.tocoo()
    mask = coo.row < coo.col
    return np.column_stack([coo.row[mask], coo.col[mask]])


def compute_edge_gradients(field, edges):
    """Compute edge gradients f_i - f_j for each edge."""
    i = edges[:, 0]
    j = edges[:, 1]
    return field[i] - field[j]


def coupling(field1, field2, edges):
    """Pearson correlation of edge gradients between two fields."""
    g1 = compute_edge_gradients(field1, edges)
    g2 = compute_edge_gradients(field2, edges)
    valid = np.isfinite(g1) & np.isfinite(g2)
    if valid.sum() < 5:
        return np.nan
    if np.nanstd(g1[valid]) < 1e-12 or np.nanstd(g2[valid]) < 1e-12:
        return np.nan
    return float(np.corrcoef(g1[valid], g2[valid])[0, 1])


def spectral_randomize(field, W, n_eigs_max=100, random_state=None):
    """Generate one spectral surrogate by random sign flips in Laplacian basis."""
    rng = np.random.default_rng(random_state)
    x = np.asarray(field, dtype=float)
    x = np.nan_to_num(x, nan=np.nanmean(x))

    N = W.shape[0]
    if N < 4:
        # tiny graph fallback
        return rng.permutation(x)

    L = graph_laplacian(W)
    k = min(n_eigs_max, max(2, N - 2))

    try:
        eigvals, eigvecs = spla.eigsh(L, k=k, which='SM')
    except Exception:
        # robust fallback
        A = L.toarray()
        eigvals_all, eigvecs_all = np.linalg.eigh(A)
        idx = np.argsort(eigvals_all)[:k]
        eigvecs = eigvecs_all[:, idx]

    coeffs = eigvecs.T @ x
    signs = rng.choice([-1.0, 1.0], size=coeffs.shape)
    coeffs_rand = coeffs * signs
    x_rand = eigvecs @ coeffs_rand

    # preserve original mean/variance scale
    x_rand = (x_rand - x_rand.mean()) / (x_rand.std() + 1e-12)
    x_ref = (x - x.mean()) / (x.std() + 1e-12)
    x_rand = x_rand * np.nanstd(x_ref) + np.nanmean(x)

    return x_rand


def spectral_null_test(field1, field2, W, edges, n_perm=500, n_eigs_max=100, random_state=0):
    """Compute observed coupling and two-sided spectral-null p-value."""
    rng = np.random.default_rng(random_state)
    obs = coupling(field1, field2, edges)
    if not np.isfinite(obs):
        return np.nan, np.array([], dtype=float), np.nan

    null_vals = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        f1_rand = spectral_randomize(
            field1,
            W,
            n_eigs_max=n_eigs_max,
            random_state=int(rng.integers(0, 1_000_000_000)),
        )
        null_vals[i] = coupling(f1_rand, field2, edges)

    null_vals = null_vals[np.isfinite(null_vals)]
    if null_vals.size == 0:
        return obs, np.array([], dtype=float), np.nan

    p = (np.sum(np.abs(null_vals) >= np.abs(obs)) + 1) / (null_vals.size + 1)
    return float(obs), null_vals, float(p)


In [ ]:
# Compute spectral-null tests per sample for key pair(s).
# Primary comparison pair: EXH vs CYTO (matches torus_CYTOlag_vs_EXH in notebook 03 exports).
pairs = [
    ("spot_exhaustion", "spot_cytotoxic", "EXH_vs_CYTO"),
    ("spot_exhaustion", "spot_tam", "EXH_vs_TAM"),
]

rows = []
for sample, ad in st_by_sample.items():
    sq.gr.spatial_neighbors(ad, coord_type="grid")
    W = ad.obsp["spatial_connectivities"].tocsr()
    edges = edge_index_from_W(W)

    for key1, key2, pair_name in pairs:
        f1 = ad.obs[key1].to_numpy(dtype=float)
        f2 = ad.obs[key2].to_numpy(dtype=float)

        obs, null_vals, p = spectral_null_test(
            f1,
            f2,
            W,
            edges,
            n_perm=N_PERM,
            n_eigs_max=N_EIGS_MAX,
            random_state=SEED,
        )

        rows.append({
            "sample": sample,
            "group": sample_groups.get(sample),
            "pair": pair_name,
            "obs_coupling": obs,
            "p_spectral_null": p,
            "n_eff_perm": int(len(null_vals)),
        })

df_spectral = pd.DataFrame(rows).sort_values(["pair", "sample"]).reset_index(drop=True)
display(df_spectral)


In [ ]:
# Load torus-null p-values from notebook-03 outputs or in-memory metrics_df.
if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame):
    torus_src = metrics_df.copy()
else:
    torus_src = None

if torus_src is None:
    candidates = [
        Path("supplement_exports/Table_S1_metrics_couplings_torus.csv"),
        Path("supplement_exports_split/Table_S1_master_metrics_baselines.csv"),
        Path("supplement_exports_v2/Table_S1_master_metrics_baselines_torus.csv"),
    ]
    for cp in candidates:
        if cp.exists():
            torus_src = pd.read_csv(cp)
            print("Loaded torus source:", cp)
            break

if torus_src is None:
    raise FileNotFoundError(
        "No torus metrics table found. Run 03_core_spatial_pipeline.ipynb first."
    )

required_torus = [
    "sample",
    "torus_CYTOlag_vs_EXH_p_torus",
    "torus_TAMlag_vs_EXH_p_torus",
]
missing_torus = [c for c in required_torus if c not in torus_src.columns]
if missing_torus:
    raise KeyError(f"Torus source missing required columns: {missing_torus}")

df_torus = torus_src[["sample", "torus_CYTOlag_vs_EXH_p_torus", "torus_TAMlag_vs_EXH_p_torus"]].copy()

df_cyto = (
    df_spectral[df_spectral["pair"] == "EXH_vs_CYTO"]
    .merge(df_torus, on="sample", how="inner")
    .rename(columns={"torus_CYTOlag_vs_EXH_p_torus": "p_torus"})
)

df_tam = (
    df_spectral[df_spectral["pair"] == "EXH_vs_TAM"]
    .merge(df_torus, on="sample", how="inner")
    .rename(columns={"torus_TAMlag_vs_EXH_p_torus": "p_torus"})
)

print("CYTO pair rows:", len(df_cyto))
print("TAM pair rows:", len(df_tam))
display(df_cyto)
display(df_tam)


In [ ]:
# Compare torus vs spectral p-values.
def _plot_null_compare(df_cmp, title, out_png):
    plt.figure(figsize=(5.5, 5))
    plt.scatter(df_cmp["p_torus"], df_cmp["p_spectral_null"])
    plt.axline((0, 0), (1, 1), linestyle="--")
    plt.xlabel("Torus p-value")
    plt.ylabel("Spectral-null p-value")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

_plot_null_compare(df_cyto, "Null model comparison: EXH vs CYTO", "fig_null_compare_exh_vs_cyto.png")
_plot_null_compare(df_tam, "Null model comparison: EXH vs TAM", "fig_null_compare_exh_vs_tam.png")

# Optional numeric summary (Spearman between p-value ranks).
rho_cyto = df_cyto[["p_torus", "p_spectral_null"]].corr(method="spearman").iloc[0, 1] if len(df_cyto) >= 2 else np.nan
rho_tam = df_tam[["p_torus", "p_spectral_null"]].corr(method="spearman").iloc[0, 1] if len(df_tam) >= 2 else np.nan

print("Spearman rho (CYTO):", rho_cyto)
print("Spearman rho (TAM):", rho_tam)
print("Saved: fig_null_compare_exh_vs_cyto.png, fig_null_compare_exh_vs_tam.png")


In [ ]:
# Export outputs.
out_dir = Path("spectral_null_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

df_spectral.to_csv(out_dir / "spectral_null_results_per_sample.csv", index=False)
df_cyto.to_csv(out_dir / "null_compare_exh_vs_cyto.csv", index=False)
df_tam.to_csv(out_dir / "null_compare_exh_vs_tam.csv", index=False)

print("Saved files:")
print("-", out_dir / "spectral_null_results_per_sample.csv")
print("-", out_dir / "null_compare_exh_vs_cyto.csv")
print("-", out_dir / "null_compare_exh_vs_tam.csv")


In [ ]:
# Build comparable vectors for quick spread check.
# Prefer EXH_vs_CYTO table; fallback to EXH_vs_TAM if needed.
if "df_cyto" in globals() and isinstance(df_cyto, pd.DataFrame) and len(df_cyto) > 0:
    _cmp = df_cyto
elif "df_tam" in globals() and isinstance(df_tam, pd.DataFrame) and len(df_tam) > 0:
    _cmp = df_tam
else:
    raise RuntimeError("Need df_cyto or df_tam from earlier cells before running this block.")

null_vals_spectral = _cmp["p_spectral_null"].dropna().to_numpy(dtype=float)
null_vals_torus = _cmp["p_torus"].dropna().to_numpy(dtype=float)

if null_vals_spectral.size == 0 or null_vals_torus.size == 0:
    raise ValueError("No p-values available to summarize.")

print("std spectral p-values:", float(np.std(null_vals_spectral)))
print("std torus p-values:", float(np.std(null_vals_torus)))


In [ ]:
from scipy.stats import combine_pvalues

# Fisher combine spectral-null p-values for EXH_vs_CYTO across samples.
pvals = df_spectral.loc[df_spectral["pair"] == "EXH_vs_CYTO", "p_spectral_null"].dropna().to_numpy(dtype=float)

if pvals.size == 0:
    raise ValueError("No EXH_vs_CYTO spectral p-values found.")

stat, p_combined = combine_pvalues(pvals, method="fisher")
print("Fisher combined statistic:", float(stat))
print("Fisher combined p-value:", float(p_combined))


In [14]:
# compute group means for EXH_vs_CYTO spectral-validated couplings
df = df_spectral[df_spectral["pair"]=="EXH_vs_TAM"]

mean_R = df[df["group"]=="R"]["obs_coupling"].mean()
mean_NR = df[df["group"]=="NR"]["obs_coupling"].mean()

print("mean_R:", mean_R)
print("mean_NR:", mean_NR)
print("diff_R_minus_NR:", mean_R - mean_NR)

mean_R: 0.022473016373344507
mean_NR: 0.026190120302999786
diff_R_minus_NR: -0.0037171039296552792


In [15]:
import numpy as np

def patient_label_permutation(values, labels, n_perm=10000):
    obs = np.mean(values[labels=="R"]) - np.mean(values[labels=="NR"])
    null = []
    for _ in range(n_perm):
        perm = np.random.permutation(labels)
        null.append(np.mean(values[perm=="R"]) - np.mean(values[perm=="NR"]))
    p = np.mean(np.abs(null) >= np.abs(obs))
    return obs, p

values = df["obs_coupling"].values
labels = df["group"].values

obs_diff, p_group = patient_label_permutation(values, labels)
print("Group diff:", obs_diff)
print("Permutation p:", p_group)

Group diff: -0.0037171039296552792
Permutation p: 0.6877
